# Smart MCQ Solver

**Objective**: The goal of this competition is to build a machine learning pipeline capable of answering complex multiple-choice questions. 

Given a question prompt and five possible options (A, B, C, D, E), the model must rank the top 3 most likely answers. The evaluation metric for the competition is **Mean Average Precision at 3 (MAP@3)**, meaning models are rewarded heavily for getting the correct answer as their #1 prediction.

## 1) Project Setup & Data Ingestion
*Initializing the environment and loading the competition datasets.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

import torch
import torch.nn.functional as F
import gc
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForMultipleChoice

import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

dtype_dict = {'prompt': str, 'A': str, 'B': str, 'C': str, 'D': str, 'E': str, 'answer': str}
train_df = pd.read_csv(train_path, dtype=dtype_dict)
test_df = pd.read_csv(test_path, dtype=dtype_dict)

if 'id' in train_df.columns:
    train_df['id'] = train_df['id'].astype(str)
if 'id' in test_df.columns:
    test_df['id'] = test_df['id'].astype(str)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}")
print("\nData Types:")
print(train_df.dtypes)

## 2) Global Data Cleaning
*Checking for missing values and duplicates across the entire dataset before applying any model-specific logic.*

In [ ]:
print("Missing Values:\n", train_df.isnull().sum())

In [ ]:
print(f"Total rows before deduplication: {len(train_df)}")
train_df.drop_duplicates(inplace=True)
print(f"Total rows after dropping exact row duplicates: {len(train_df)}")

### Conclusion: 
- There were no missing values so we left it as it is.
- We dropped exact row duplicates to prevent data leakage. However, we kept partial duplicates (where the prompt is identical but the questions/options differ) because this is common in reading comprehension datasets.

## 3) Exploratory Data Analysis (EDA)
*Visualizing class distributions, text lengths, and option variances.*

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x='answer', order=['A', 'B', 'C', 'D', 'E'], palette='viridis')
plt.title('Target Class Distribution')
plt.show()

In [ ]:
train_df['prompt_word_count'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 4))
sns.histplot(data=train_df, x='prompt_word_count', bins=30, kde=True, color='blue')
plt.title('Prompt Length (Word Count) Distribution')
plt.show()

In [ ]:
option_lengths = []
for opt in ['A', 'B', 'C', 'D', 'E']:
    lengths = train_df[opt].apply(lambda x: len(str(x).split()))
    for l in lengths:
        option_lengths.append({'Option': opt, 'Word Count': l})
opt_df = pd.DataFrame(option_lengths)

plt.figure(figsize=(8, 4))
sns.boxplot(data=opt_df, x='Option', y='Word Count', palette='pastel')
plt.title('Word Counts Across Options')
plt.ylim(0, 50)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_df, x='answer', y='prompt_word_count', order=['A', 'B', 'C', 'D', 'E'], palette='muted')
plt.title('Prompt Length vs. Correct Answer')
plt.show()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
train_df['prompt_token_count'] = train_df['prompt'].apply(lambda x: len(tokenizer.encode(str(x))))

plt.figure(figsize=(8, 4))
sns.histplot(data=train_df, x='prompt_token_count', bins=30, kde=True, color='red')
plt.axvline(x=256, color='black', linestyle='--', label='MiniLM Max Length (256)')
plt.title('Prompt Length (Tokenizer Tokens) Distribution')
plt.legend()
plt.show()

overflow_count = (train_df['prompt_token_count'] > 256).sum()
print(f"Warning: {overflow_count} prompts exceed 256 tokens and are being silently truncated by MiniLM!")

## 4) Shared Utilities & Results Tracking
*Foundational helper functions for evaluation, data formatting, and maintaining a strict validation split.*

### Validation Split & Central Results Tracker
*To ensure we can fairly compare all future models, we will create a dedicated tracking DataFrame (`val_results_df`) that holds the ground truth and will store predictions from every model we build.*

In [ ]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])

val_results_df = val_split[['id', 'answer']].copy() if 'id' in val_split.columns else pd.DataFrame({'answer': val_split['answer']})

print(f"Train Split: {len(train_split)} rows")
print(f"Validation Split: {len(val_split)} rows")
print("Central Results Tracker initialized.")

### Text Formatting Helpers
*Extracting options and building concatenated strings cleanly so we don't have messy inline loops later.*

In [ ]:
def get_options_list(row):
    return [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]

def format_multiple_choice(prompt, option):
    return f"{str(prompt).strip()} {str(option).strip()}"

def format_transformer_option(option_text, option_letter):
    return f"({option_letter}) {str(option_text).strip()}"

### Evaluation Metric (MAP@3)

In [ ]:
def calculate_map3(predictions, ground_truth):
    scores = []
    for pred_str, gt in zip(predictions, ground_truth):
        pred_list = pred_str.strip().split()
        score = 0.0
        num_correct = 0
        for k, label in enumerate(pred_list[:3], start=1):
            if label == gt:
                num_correct += 1
                score += num_correct / k
        scores.append(score)
    return np.mean(scores)

dummy_preds = ['A B C', 'D A E', 'C B A']
dummy_truths = ['A', 'A', 'C']
print(f"Sanity Check MAP@3 Score: {calculate_map3(dummy_preds, dummy_truths)}")

## 5) Model 1 (TF-IDF Baseline)
*Unlike Transformer models which rely on case and punctuation for semantic context, TF-IDF is a lexical Bag-of-Words model. We must aggressively normalize the text specifically for this model, and then compute cosine similarity.*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text_for_tfidf(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

print("Original:", train_df['prompt'].iloc[0][:100])
print("Cleaned TF-IDF:", clean_text_for_tfidf(train_df['prompt'].iloc[0][:100]))

In [ ]:
tfidf_preds = []

for idx, row in val_split.iterrows():
    prompt_clean = clean_text_for_tfidf(row['prompt'])
    
    raw_opts = get_options_list(row)
    opts_clean = [clean_text_for_tfidf(opt) for opt in raw_opts]
    
    vec = TfidfVectorizer()
    try:
        tfidf_matrix = vec.fit_transform([prompt_clean] + opts_clean)
        sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
        
        top_indices = np.argsort(sims)[::-1][:3]
        pred_str = " ".join([['A', 'B', 'C', 'D', 'E'][i] for i in top_indices])
    except ValueError:
        pred_str = "B A C"
        
    tfidf_preds.append(pred_str)

val_results_df['tfidf_predictions'] = tfidf_preds

tfidf_map3 = calculate_map3(val_results_df['tfidf_predictions'], val_results_df['answer'])
print(f"TF-IDF MAP@3 Score: {tfidf_map3:.4f}")

### Conclusion & Failed Experiment:
Our initial naive TF-IDF baseline achieved a MAP@3 of **`0.3117`**. 
We attempted to improve this by removing common English stopwords, assuming they were adding noise to the cosine similarity calculations. However, the score aggressively dropped to **`~0.27`**!

**Why?** In the context of reading comprehension and MCQs, seemingly "noisy" words like *"not"*, *"is"*, or *"because"* actually carry critical semantic signals required to match the prompt to the exact option. Removing them destroyed the context. We have kept the failed code out of our utilities to maintain the higher 0.3117 baseline, but this proves that lexical matching (TF-IDF) is far too brittle for this task. We need semantic embeddings.

## 6) Model 2 (MiniLM Semantic Embeddings)
*Unlike TF-IDF, semantic transformer models (like MiniLM or BERT) depend heavily on exact sentence structure, casing, and punctuation to extract meaning. We will pass the raw strings directly into a pretrained huggingface model to compute semantic similarity between the prompt and the options.*

In [ ]:
minilm_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("Model 'all-MiniLM-L6-v2' loaded successfully onto:", device)

In [ ]:
minilm_preds = []
letters = ['A', 'B', 'C', 'D', 'E']

for idx, row in val_split.iterrows():
    raw_prompt = str(row['prompt']).strip()
    raw_opts = get_options_list(row)
    
    formatted_opts = [format_transformer_option(opt, letter) for opt, letter in zip(raw_opts, letters)]
    
    with torch.no_grad():
        prompt_emb = minilm_model.encode(raw_prompt, convert_to_tensor=True)
        opts_emb = minilm_model.encode(formatted_opts, convert_to_tensor=True)
        
        sims = util.cos_sim(prompt_emb, opts_emb).cpu().numpy()[0]
        
    top_indices = np.argsort(sims)[::-1][:3]
    pred_str = " ".join([letters[i] for i in top_indices])
    
    minilm_preds.append(pred_str)

val_results_df['minilm_predictions'] = minilm_preds

minilm_map3 = calculate_map3(val_results_df['minilm_predictions'], val_results_df['answer'])
print(f"MiniLM MAP@3 Score: {minilm_map3:.4f}")

## 7) Model 3 (Fine-Tuned Ensemble & Grid Search)
*Zero-shotting base models often performs worse than semantic similarity models. To unlock the true power of Heavyweight Transformers (DeBERTa and RoBERTa), we explicitly fine-tune them on the 80% training split and run a mathematical grid search on the 20% validation split to discover the optimal ensembling weight.*

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from dataclasses import dataclass
from typing import Optional, Union

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


ft_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
ft_model = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-small")


In [ ]:
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)
train_df_80 = train_split.copy()
train_df_80['label'] = train_df_80['answer'].map(label_map)
val_df = val_split.copy()
val_df['label'] = val_df['answer'].map(label_map)

def preprocess_function(examples):
    prompts = [[context] * 5 for context in examples['prompt']]
    options = [[f"(A) {examples['A'][i]}", f"(B) {examples['B'][i]}", f"(C) {examples['C'][i]}", f"(D) {examples['D'][i]}", f"(E) {examples['E'][i]}"] for i in range(len(examples['prompt']))]
    prompts = sum(prompts, [])
    options = sum(options, [])
    tokenized = ft_tokenizer(prompts, options, truncation=True, padding=False)
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

train_ds = Dataset.from_pandas(train_df_80[['prompt', 'A', 'B', 'C', 'D', 'E', 'label']])
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E'])

val_ds = Dataset.from_pandas(val_df[["prompt", "A", "B", "C", "D", "E", "label"]])
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=["prompt", "A", "B", "C", "D", "E"])


In [ ]:
import torch.nn.functional as F

def train_eval_model(model_name):
    print(f"\nTraining {model_name} on 80% split for Grid Search")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMultipleChoice.from_pretrained(model_name)
    
    def prep_fn(examples):
        prompts = [[context] * 5 for context in examples['prompt']]
        options = [[f"(A) {examples['A'][i]}", f"(B) {examples['B'][i]}", f"(C) {examples['C'][i]}", f"(D) {examples['D'][i]}", f"(E) {examples['E'][i]}"] for i in range(len(examples['prompt']))]
        tokenized = tokenizer(sum(prompts, []), sum(options, []), truncation=True, padding=False)
        return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
        
    tok_train = train_ds.map(prep_fn, batched=True, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E'])
    tok_val = val_ds.map(prep_fn, batched=True, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E'])
    
    args = TrainingArguments(
        output_dir=f"./eval_{model_name.replace('/', '_')}",
        learning_rate=1e-5,
        warmup_steps=100,
        adam_epsilon=1e-6,
        weight_decay=0.01,
        save_strategy="no",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        processing_class=tokenizer,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    )
    
    trainer.train()
    
    print(f"Running inference on 20% validation split for {model_name}...")
    preds = trainer.predict(tok_val)
    probs = F.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
    return probs

d_probs_val = train_eval_model("microsoft/deberta-v3-small")
r_probs_val = train_eval_model("roberta-base")

best_w = 0.50 # Fallback initialization, overwritten by grid search
best_map3 = 0.0
letters = ['A', 'B', 'C', 'D', 'E']

print("\nRunning Grid Search for optimal ensemble weight (DeBERTa : RoBERTa)")
for w in np.arange(0.1, 1.0, 0.05):
    ensembled_probs = (d_probs_val * w) + (r_probs_val * (1.0 - w))
    
    val_preds_list = []
    for p in ensembled_probs:
        top_indices = np.argsort(p)[::-1][:3]
        val_preds_list.append(" ".join([letters[idx] for idx in top_indices]))
        
    current_map3 = calculate_map3(val_preds_list, val_results_df['answer'])
    if current_map3 > best_map3:
        best_map3 = current_map3
        best_w = w

print(f"Optimal DeBERTa Weight: {best_w:.2f} (RoBERTa: {1-best_w:.2f})")
print(f"Optimal Fine-Tuned Ensemble MAP@3 Score: {best_map3:.4f}")

ft_weighted_map3 = best_map3
FINAL_DEBERTA_WEIGHT = best_w
FINAL_ROBERTA_WEIGHT = 1.0 - best_w

# Save the optimal predictions to our tracker
best_probs = (d_probs_val * best_w) + (r_probs_val * (1.0 - best_w))
best_preds = []
for p in best_probs:
    top_indices = np.argsort(p)[::-1][:3]
    best_preds.append(" ".join([letters[idx] for idx in top_indices]))
val_results_df['ft_weighted_ensemble_predictions'] = best_preds


## 8) Final Model Comparison & Tracking
*Aggregating the performance of all tested pipelines.*

In [ ]:
comparison_data = [
    {'Model': 'Model 1 (TF-IDF Lexical)', 'MAP@3': tfidf_map3},
    {'Model': 'Model 2 (MiniLM Semantic)', 'MAP@3': minilm_map3},
    {'Model': 'Model 3 (Fine-Tuned Ensemble)', 'MAP@3': ft_weighted_map3}
]
comp_df = pd.DataFrame(comparison_data)
display(comp_df)

### Different model's answer to same questions

In [ ]:
display(val_results_df.head())

### Model Performance Visualization

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=comp_df, x='MAP@3', y='Model', palette='coolwarm')
plt.title('Validation MAP@3 Score by Model')
plt.xlim(0, 1.0)

for index, value in enumerate(comp_df['MAP@3']):
    plt.text(value + 0.01, index, f"{value:.4f}", va='center')

plt.show()


## 9) Final Kaggle Submission Pipeline
*Retraining the dynamically weighted ensemble on the full dataset (100%) to maximize accuracy on the unseen test set.*

In [ ]:
print(f"Targeting: {comp_df.loc[comp_df['MAP@3'].idxmax()]['Model']}")

def train_final_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMultipleChoice.from_pretrained(model_name)
    
    full_train_ds = Dataset.from_pandas(train_df[['prompt', 'A', 'B', 'C', 'D', 'E', 'label']])
    
    def prep_fn(examples):
        prompts = [[context] * 5 for context in examples['prompt']]
        options = [[f"(A) {examples['A'][i]}", f"(B) {examples['B'][i]}", f"(C) {examples['C'][i]}", f"(D) {examples['D'][i]}", f"(E) {examples['E'][i]}"] for i in range(len(examples['prompt']))]
        tokenized = tokenizer(sum(prompts, []), sum(options, []), truncation=True, padding=False)
        return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
        
    tokenized_full = full_train_ds.map(prep_fn, batched=True, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E'])
    
    args = TrainingArguments(
        output_dir=f"./final_{model_name.replace('/', '_')}",
        learning_rate=1e-5,
        warmup_steps=100,
        adam_epsilon=1e-6,
        weight_decay=0.01,
        gradient_accumulation_steps=2,
        save_strategy="no",
        per_device_train_batch_size=8,
        num_train_epochs=5,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_full,
        processing_class=tokenizer,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    )
    
    trainer.train()
    return tokenizer, model

final_deb_tok, final_deb_model = train_final_model("microsoft/deberta-v3-small")
final_rob_tok, final_rob_model = train_final_model("roberta-base")


In [ ]:
import torch.nn.functional as F

submission_preds = []

final_deb_model.eval()
final_rob_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_deb_model.to(device)
final_rob_model.to(device)

letters = ['A', 'B', 'C', 'D', 'E']
batch_size = 8

for i in range(0, len(test_df), batch_size):
    batch_df = test_df.iloc[i:i+batch_size]
    
    prompts = sum([[context] * 5 for context in batch_df['prompt']], [])
    options = sum([[f"(A) {row['A']}", f"(B) {row['B']}", f"(C) {row['C']}", f"(D) {row['D']}", f"(E) {row['E']}"] for _, row in batch_df.iterrows()], [])
    
    d_inputs = final_deb_tok(prompts, options, truncation=True, padding=True, return_tensors="pt")
    d_inputs = {k: v.view(len(batch_df), 5, -1).to(device) for k, v in d_inputs.items()}
    
    r_inputs = final_rob_tok(prompts, options, truncation=True, padding=True, return_tensors="pt")
    r_inputs = {k: v.view(len(batch_df), 5, -1).to(device) for k, v in r_inputs.items()}
    
    with torch.no_grad():
        d_probs = F.softmax(final_deb_model(**d_inputs).logits, dim=-1).cpu().numpy()
        r_probs = F.softmax(final_rob_model(**r_inputs).logits, dim=-1).cpu().numpy()
        
    ensembled_probs = (d_probs * FINAL_DEBERTA_WEIGHT) + (r_probs * FINAL_ROBERTA_WEIGHT)
    
    for p in ensembled_probs:
        top_indices = np.argsort(p)[::-1][:3]
        pred_str = " ".join([letters[idx] for idx in top_indices])
        submission_preds.append(pred_str)


In [ ]:
# Format the final predictions into the dataframe structure required by Kaggle
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)

# Quick sanity check before uploading
submission_df.head()
